# Ch05 -- 抽樣分佈與中央極限定理 (Central Limit Theorem, CLT)

> **預計時間：1.5 小時**

## 學習目標

1. 理解**母體分佈 (Population Distribution)**、**樣本分佈 (Sample Distribution)** 與**抽樣分佈 (Sampling Distribution)** 三者的區別
2. 掌握**中央極限定理 (Central Limit Theorem)** 的核心陳述與直覺意義
3. 透過模擬驗證 CLT，觀察樣本均值的分佈如何隨樣本大小趨近常態
4. 理解**標準誤 (Standard Error)** 的定義與實務意涵
5. 區分**大數法則 (Law of Large Numbers)** 與 CLT 的不同角色
6. 認識 CLT 作為推論統計基石的實務意義

## 前置章節

本章建立在以下基礎之上：
- **Ch01 ~ Ch03**：假設檢定 (Hypothesis Testing)、卡方檢定 (Chi-Squared Tests)、變異數分析 (ANOVA)
- **Ch04**：A/B 測試流程 (A/B Testing Procedure)

如果尚未完成上述章節，建議先行複習。

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 載入 Titanic 資料集，以 Fare 欄位作為偏態母體
df = pd.read_csv('datasets/titanic_train.csv')
fare = df['Fare'].dropna()

print(f'Fare 樣本數: {len(fare)}')
print(f'Fare 平均值: {fare.mean():.2f}')
print(f'Fare 標準差: {fare.std():.2f}')
print(f'Fare 偏態係數 (Skewness): {fare.skew():.2f}')

---

## Part A -- 抽樣分佈概念 (Sampling Distribution)

### 三種分佈的區別

| 名稱 | 英文 | 描述 |
|------|------|------|
| **母體分佈** | Population Distribution | 整個母體中所有觀測值的分佈 |
| **樣本分佈** | Sample Distribution | 從母體中抽出的一組樣本的分佈 |
| **抽樣分佈** | Sampling Distribution | 某個統計量（如樣本均值 $\bar{X}$）在重複抽樣下的分佈 |

**關鍵觀念**：
- 母體分佈和樣本分佈描述的是**個別觀測值**
- 抽樣分佈描述的是**統計量**（例如平均值、比例、變異數等）
- CLT 告訴我們：不論母體分佈的形狀為何，只要樣本數夠大，**樣本均值的抽樣分佈**會趨近常態分佈 (Normal Distribution)

In [ ]:
# 畫 Fare 的母體分佈（直方圖 Histogram）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：原始分佈
axes[0].hist(fare, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(fare.mean(), color='red', ls='--', lw=2, label=f'Mean = {fare.mean():.2f}')
axes[0].axvline(fare.median(), color='orange', ls='--', lw=2, label=f'Median = {fare.median():.2f}')
axes[0].set_title('Fare 母體分佈 (Population Distribution)', fontsize=13)
axes[0].set_xlabel('Fare')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# 右圖：對數轉換後
log_fare = np.log1p(fare)
axes[1].hist(log_fare, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_title('log(1 + Fare) 分佈', fontsize=13)
axes[1].set_xlabel('log(1 + Fare)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print('觀察：Fare 呈現明顯右偏 (Right-Skewed)，這正好適合用來驗證 CLT。')

**小結**：Titanic 的 Fare 欄位呈現明顯的右偏分佈，完全不是常態分佈。  
接下來我們將用 CLT 模擬來展示：即使母體分佈如此偏態，樣本均值的分佈依然會趨近常態。

---

## Part B -- 中央極限定理模擬 (CLT Simulation)

### CLT 的正式陳述

設 $X_1, X_2, \ldots, X_n$ 為來自母體（平均值 $\mu$、變異數 $\sigma^2$）的獨立同分佈 (i.i.d.) 隨機變數，則當 $n$ 足夠大時：

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i \xrightarrow{d} N\left(\mu, \frac{\sigma^2}{n}\right)$$

標準化後：

$$Z = \frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} N(0, 1)$$

**白話版**：不管原始資料長什麼樣子，只要樣本數 $n$ 夠大，「很多次取樣的平均值」會形成一個鐘形曲線（常態分佈）。

In [ ]:
# CLT 模擬：對 Fare 進行重複抽樣，觀察樣本均值的分佈
np.random.seed(42)
sample_sizes = [5, 10, 30, 100]
n_simulations = 1000

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, n in zip(axes.flatten(), sample_sizes):
    means = [fare.sample(n, replace=True).mean() for _ in range(n_simulations)]
    ax.hist(means, bins=30, edgecolor='black', alpha=0.7, color='steelblue', density=True)
    ax.axvline(fare.mean(), color='red', ls='--', lw=2, label=f'Population Mean = {fare.mean():.1f}')
    ax.set_title(f'n = {n}  (skew = {pd.Series(means).skew():.2f})', fontsize=12)
    ax.set_xlabel('Sample Mean of Fare')
    ax.legend(fontsize=8)

plt.suptitle('CLT 模擬：Fare 的抽樣分佈 (Sampling Distribution of Mean)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 觀察結果

- **n = 5**：抽樣分佈仍然呈現明顯右偏，離常態分佈還很遠
- **n = 10**：偏態減少，但仍可見不對稱
- **n = 30**：分佈已接近鐘形曲線，CLT 效果明顯
- **n = 100**：幾乎完美的常態分佈

> **口訣：n >= 30 就夠安心**  
> 這是統計學中常見的經驗法則 (Rule of Thumb)。  
> 但要注意：母體分佈越偏態，需要的 n 就越大。

In [ ]:
# 常態性檢定 (Normality Test)：用 Shapiro-Wilk Test 驗證
np.random.seed(42)
print('Shapiro-Wilk Test for Sampling Distribution of Mean:')
print(f'{"n":>5} | {"W statistic":>12} | {"p-value":>10} | {"Normal?":>8}')
print('-' * 50)

for n in sample_sizes:
    means = [fare.sample(n, replace=True).mean() for _ in range(n_simulations)]
    w_stat, p_val = stats.shapiro(np.random.choice(means, 200))  # 取 200 個避免 Shapiro 樣本限制
    normal = 'Yes' if p_val > 0.05 else 'No'
    print(f'{n:>5} | {w_stat:>12.4f} | {p_val:>10.4f} | {normal:>8}')

In [ ]:
# Q-Q Plot：視覺化驗證常態性
np.random.seed(42)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, n in zip(axes.flatten(), sample_sizes):
    means = [fare.sample(n, replace=True).mean() for _ in range(n_simulations)]
    stats.probplot(means, dist='norm', plot=ax)
    ax.set_title(f'Q-Q Plot: n = {n}', fontsize=12)

plt.suptitle('Q-Q Plot 驗證抽樣分佈的常態性', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**解讀 Q-Q Plot**：
- 如果資料點落在對角線上，表示分佈接近常態
- n = 5 時尾端明顯偏離；n = 30 以上幾乎完全貼合

---

## Part C -- 標準誤 (Standard Error, SE)

### 標準誤的定義

標準誤 (Standard Error) 是**抽樣分佈的標準差**，衡量統計量（如樣本均值）的抽樣變異程度。

$$SE = \frac{\sigma}{\sqrt{n}}$$

其中 $\sigma$ 是母體標準差，$n$ 是樣本大小。

**重要性質**：
- SE 隨 $n$ 增大而減小（精度提高）
- SE 與 $\sqrt{n}$ 成反比，不是與 $n$ 成反比

> **口訣：樣本大四倍，精度好兩倍**  
> 因為 $\frac{\sigma}{\sqrt{4n}} = \frac{1}{2} \cdot \frac{\sigma}{\sqrt{n}}$

In [ ]:
# 模擬不同 n 的 SE，與理論值比較
np.random.seed(42)
n_values = np.arange(5, 201, 5)
simulated_se = []
theoretical_se = []

pop_std = fare.std()  # 母體標準差

for n in n_values:
    means = [fare.sample(n, replace=True).mean() for _ in range(1000)]
    simulated_se.append(np.std(means))
    theoretical_se.append(pop_std / np.sqrt(n))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(n_values, simulated_se, 'o', markersize=4, label='Simulated SE', color='steelblue')
ax.plot(n_values, theoretical_se, '-', lw=2, label=r'Theoretical SE = $\sigma / \sqrt{n}$', color='red')
ax.set_xlabel('Sample Size (n)', fontsize=12)
ax.set_ylabel('Standard Error', fontsize=12)
ax.set_title('Standard Error vs Sample Size', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 驗證「樣本大四倍，精度好兩倍」
se_n25 = pop_std / np.sqrt(25)
se_n100 = pop_std / np.sqrt(100)

print(f'SE (n=25):  {se_n25:.4f}')
print(f'SE (n=100): {se_n100:.4f}')
print(f'Ratio:      {se_n25 / se_n100:.1f}x')
print()
print('n 從 25 增加到 100（4 倍），SE 減半 -- 精度提升 2 倍。')

### SE 的實務意義

- **信賴區間 (Confidence Interval)** 的寬度與 SE 成正比：SE 越小，CI 越窄，估計越精確
- **假設檢定**的檢定統計量（如 t 值、z 值）都用到 SE
- 增加樣本數有**邊際遞減效應**：從 n=10 到 n=40 的改善，遠大於從 n=100 到 n=130 的改善

---

## Part D -- 大數法則 (Law of Large Numbers, LLN)

### LLN vs CLT 的區別

這兩個定理經常被混淆，但它們回答的是**不同的問題**：

| | 大數法則 (LLN) | 中央極限定理 (CLT) |
|---|---|---|
| **問什麼** | $\bar{X}_n$ 會收斂到哪裡？ | $\bar{X}_n$ 的分佈長什麼樣子？ |
| **結論** | $\bar{X}_n \to \mu$（幾乎確定） | $\bar{X}_n \sim N(\mu, \sigma^2/n)$（近似） |
| **白話** | 樣本越大，平均值越接近真值 | 不管原始分佈，平均值的分佈趨近常態 |
| **關注** | 收斂的**目標** | 收斂過程中的**形狀** |

In [ ]:
# 累積平均軌跡圖 -- 展示大數法則 (LLN)
np.random.seed(42)

# 從 Fare 中隨機抽取 500 個觀測值
random_draws = fare.sample(500, replace=True).values
cumulative_means = np.cumsum(random_draws) / np.arange(1, 501)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, 501), cumulative_means, color='steelblue', lw=1.5, label='Cumulative Mean')
ax.axhline(fare.mean(), color='red', ls='--', lw=2, label=f'Population Mean = {fare.mean():.2f}')
ax.fill_between(range(1, 501),
                fare.mean() - 5, fare.mean() + 5,
                alpha=0.15, color='red', label=r'$\mu \pm 5$')
ax.set_xlabel('Number of Observations (n)', fontsize=12)
ax.set_ylabel('Cumulative Mean of Fare', fontsize=12)
ax.set_title('大數法則 (LLN)：累積平均值收斂至母體平均值', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 多條軌跡疊加，展示 LLN 的穩定性
np.random.seed(123)
n_paths = 20

fig, ax = plt.subplots(figsize=(12, 5))

for i in range(n_paths):
    draws = fare.sample(300, replace=True).values
    cum_mean = np.cumsum(draws) / np.arange(1, 301)
    ax.plot(range(1, 301), cum_mean, alpha=0.3, lw=0.8)

ax.axhline(fare.mean(), color='red', ls='--', lw=2, label=f'Population Mean = {fare.mean():.2f}')
ax.set_xlabel('Number of Observations (n)', fontsize=12)
ax.set_ylabel('Cumulative Mean', fontsize=12)
ax.set_title(f'大數法則：{n_paths} 條獨立軌跡皆收斂至 Population Mean', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**觀察**：
- 初期（n 小）時，各條軌跡波動劇烈
- 隨著 n 增大，所有軌跡逐漸收斂到母體平均值（紅色虛線）
- 這就是**大數法則**的精髓：樣本越大，估計越準

---

## Part E -- 實務意義 (Practical Significance)

### CLT 是推論統計的基石

為什麼 CLT 如此重要？因為幾乎所有推論統計方法都依賴它：

1. **信賴區間 (Confidence Intervals)**  
   $\bar{X} \pm z_{\alpha/2} \cdot \frac{s}{\sqrt{n}}$  
   這個公式成立的前提就是 CLT（$\bar{X}$ 近似常態）

2. **假設檢定 (Hypothesis Testing)**  
   z 檢定和 t 檢定的檢定統計量都假設 $\bar{X}$ 近似常態

3. **A/B 測試 (A/B Testing)**  
   比較兩組均值差異時，CLT 讓我們可以用常態分佈來計算 p 值

4. **迴歸分析 (Regression Analysis)**  
   OLS 估計量的抽樣分佈也受 CLT 保證

### CLT 的限制

- 需要**有限變異數** ($\sigma^2 < \infty$)：柯西分佈 (Cauchy Distribution) 等重尾分佈不適用
- 需要**獨立性**：時間序列等有自相關的資料需要特殊處理
- 需要**足夠大的 n**：越偏態的分佈，需要越大的 n

### 知識補給站 -- Bootstrap 方法

當母體分佈未知、樣本數不夠大、或統計量的抽樣分佈難以推導時，可以使用 **Bootstrap** 方法：

1. 從原始樣本中**有放回地 (with replacement)** 重新抽樣，產生 Bootstrap 樣本
2. 對每個 Bootstrap 樣本計算感興趣的統計量
3. 重複上千次，用這些統計量的分佈來估計抽樣分佈

Bootstrap 的精神與 CLT 模擬類似，但它**不需要假設母體分佈**，是一種非參數 (Nonparametric) 方法。

In [ ]:
# Bootstrap 信賴區間範例
np.random.seed(42)
n_bootstrap = 5000
sample_data = fare.sample(50, replace=False, random_state=42)

bootstrap_means = [
    sample_data.sample(len(sample_data), replace=True).mean()
    for _ in range(n_bootstrap)
]

ci_lower = np.percentile(bootstrap_means, 2.5)
ci_upper = np.percentile(bootstrap_means, 97.5)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(bootstrap_means, bins=50, edgecolor='black', alpha=0.7, color='steelblue', density=True)
ax.axvline(ci_lower, color='orange', ls='--', lw=2, label=f'2.5th percentile = {ci_lower:.2f}')
ax.axvline(ci_upper, color='orange', ls='--', lw=2, label=f'97.5th percentile = {ci_upper:.2f}')
ax.axvline(fare.mean(), color='red', ls='--', lw=2, label=f'Population Mean = {fare.mean():.2f}')
ax.set_title('Bootstrap 95% Confidence Interval for Mean Fare', fontsize=14)
ax.set_xlabel('Bootstrap Sample Mean')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Bootstrap 95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]')
print(f'Population Mean:   {fare.mean():.2f}')

---

## 重點整理 (Key Takeaways)

| 概念 | 重點 |
|------|------|
| **抽樣分佈** | 統計量在重複抽樣下的分佈，不是個別觀測值的分佈 |
| **CLT** | 不論母體分佈形狀，$\bar{X}$ 的抽樣分佈在 $n$ 夠大時趨近常態 |
| **標準誤 SE** | $SE = \sigma / \sqrt{n}$，衡量估計的精度 |
| **大數法則 LLN** | $\bar{X}_n \to \mu$，關注收斂目標 |
| **CLT vs LLN** | CLT 關注分佈形狀，LLN 關注收斂目標 |
| **n >= 30** | 經驗法則，但偏態母體可能需要更大的 n |
| **Bootstrap** | 不需要分佈假設的非參數重抽樣方法 |

---

## 練習題 (Exercises)

### 練習 1：其他統計量的抽樣分佈
- 將上面的 CLT 模擬改為計算**中位數 (Median)** 而非平均值
- 中位數的抽樣分佈是否也趨近常態？需要多大的 n？

### 練習 2：不同母體分佈
- 使用 `np.random.exponential(scale=2, size=10000)` 產生指數分佈 (Exponential Distribution) 母體
- 對這個母體進行 CLT 模擬，觀察 n = 5, 10, 30, 100 的差異

### 練習 3：SE 與信賴區間
- 從 Fare 中抽取 n = 20 的樣本，計算 95% 信賴區間
- 重複抽取 100 次，檢查有多少個信賴區間包含真正的母體平均值
- 這個比例是否接近 95%？

### 練習 4：Bootstrap vs CLT
- 從 Fare 中抽取 n = 15 的小樣本
- 分別用 CLT 方法和 Bootstrap 方法建構 95% CI
- 比較兩種方法的結果

In [ ]:
# 練習區 -- 在此撰寫你的程式碼



---

## 導覽列 (Navigation)

| 上一章 | 目前 | 下一章 |
|--------|------|--------|
| [Ch04 A/B Testing Procedure](04_AB_testing_procedure.ipynb) | **Ch05 抽樣分佈與中央極限定理** | Coming Soon |

[Back to Index](../)